# 01_data_collection_mop.ipynb
## Project Title: Traffic Accident Risk Prediction (TARP)
**Unit:** SIT782  
**Prepared by:** Subathira Thinakaran  

**Project Team:**  
- Suba (225094537)  
- Burhan (224802775)  
- Khalid (224696667)  

**Task:** Clean MOP Pedestrian and Bicycle Data (Week 2 of 8)

## Objective
This notebook cleans the pedestrian and bicycle datasets collected from the City of Melbourne Open Data API. The cleaning process includes handling missing values, standardising formats, preparing time-related fields, and aggregating mobility counts by hour for later integration and analysis.

## Datasets Used


## Output


## Imports and Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import ast

In [2]:
from google.colab import files
uploaded = files.upload()

Saving bicycle.csv to bicycle.csv
Saving pedestrian.csv to pedestrian.csv


## Load Raw Datasets

Load the pedestrian and bicycle datasets collected from the previous data collection stage.  
This step verifies that the datasets are available and ready for cleaning.

In [3]:
# -----------------------------
# 1. Load raw datasets
# -----------------------------
from pathlib import Path

CLEAN_DATA_DIR = Path("data/clean")
CLEAN_DATA_DIR.mkdir(parents=True, exist_ok=True)

pedestrian_file = Path("pedestrian.csv")
bicycle_file = Path("bicycle.csv")

pedestrian_df = pd.read_csv(pedestrian_file)
bicycle_df = pd.read_csv(bicycle_file)

print("Pedestrian raw shape:", pedestrian_df.shape)
print("Bicycle raw shape:", bicycle_df.shape)

Pedestrian raw shape: (5000, 9)
Bicycle raw shape: (352, 82)


### Observation

The pedestrian dataset contains 5000 records with a relatively simple structure, while the bicycle dataset contains 352 records with a significantly higher number of features.  

This confirms that both datasets have been successfully loaded and are ready for further inspection and cleaning.

## Initial Inspection

Inspect dataset structure and identify data quality issues such as missing values and unnecessary features.

In [4]:
# -----------------------------
# 2. Initial inspection
# -----------------------------
print("Pedestrian columns:")
print(list(pedestrian_df.columns))

print("\nBicycle columns:")
print(list(bicycle_df.columns))

print("\nPedestrian missing values:")
print(pedestrian_df.isna().sum())

print("\nTop bicycle missing values:")
print(bicycle_df.isna().sum().sort_values(ascending=False).head(15))

Pedestrian columns:
['id', 'location_id', 'sensing_date', 'hourday', 'direction_1', 'direction_2', 'pedestriancount', 'sensor_name', 'location']

Bicycle columns:
['state', 'electorate', 'site_id', 'latitude', 'longitude', 'legs', 'description', 'layout_1', 'layout_1_enter', 'layout_2', 'layout_2_enter', 'layout_3', 'layout_3_enter', 'layout_4', 'layout_4_enter', 'layout_5', 'layout_5_enter', 'layout_6', 'layout_6_enter', 'leg1_2', 'leg1_3', 'leg1_4', 'leg1_5', 'leg1_6', 'leg2_1', 'leg2_3', 'leg2_4', 'leg2_5', 'leg2_6', 'leg3_1', 'leg3_2', 'leg3_4', 'leg3_5', 'leg3_6', 'leg4_1', 'leg4_2', 'leg4_3', 'leg4_5', 'leg4_6', 'leg5_1', 'leg5_2', 'leg5_3', 'leg5_4', 'leg5_6', 'leg6_1', 'leg6_2', 'leg6_3', 'leg6_4', 'leg6_5', 'leg1_enter', 'leg1_exit', 'leg1_total', 'leg2_enter', 'leg2_exit', 'leg2_total', 'leg3_enter', 'leg3_exit', 'leg3_total', 'leg4_enter', 'leg4_exit', 'leg4_total', 'leg5_enter', 'leg5_exit', 'leg5_total', 'leg6_enter', 'leg6_exit', 'leg6_total', 'female', 'male', 'not_known

### Observation

The pedestrian dataset is relatively clean, with no missing values and a simple structure focused on time, location, and pedestrian counts.

In contrast, the bicycle dataset is more complex and contains several columns with all values missing. These fully empty columns do not provide useful information and will be removed during the cleaning stage.

## Clean Pedestrian Dataset

This step standardises column names, converts date and hour fields into appropriate formats, and checks for duplicate records.

In [5]:
# -----------------------------
# 3. Clean pedestrian dataset
# -----------------------------
pedestrian_clean = pedestrian_df.copy()
pedestrian_clean.columns = pedestrian_clean.columns.str.strip().str.lower()

# Convert date column
pedestrian_clean["sensing_date"] = pd.to_datetime(
    pedestrian_clean["sensing_date"],
    errors="coerce"
)

# Ensure hour is numeric
pedestrian_clean["hourday"] = pd.to_numeric(
    pedestrian_clean["hourday"],
    errors="coerce"
)

# Check duplicates
print("Duplicate pedestrian rows:", pedestrian_clean.duplicated().sum())

print("\nPedestrian data types:")
print(pedestrian_clean.dtypes)

pedestrian_clean.head()

Duplicate pedestrian rows: 0

Pedestrian data types:
id                          int64
location_id                 int64
sensing_date       datetime64[ns]
hourday                     int64
direction_1                 int64
direction_2                 int64
pedestriancount             int64
sensor_name                object
location                   object
dtype: object


,id,location_id,sensing_date,hourday,direction_1,direction_2,pedestriancount,sensor_name,location
0,661720250602,66,2025-06-02,17,1215,1516,2731,QVN_T,"{'lon': 144.96444294, 'lat': -37.81057846}"
1,661320240712,66,2024-07-12,13,1406,1841,3247,QVN_T,"{'lon': 144.96444294, 'lat': -37.81057846}"
2,661120250310,66,2025-03-10,11,763,893,1656,QVN_T,"{'lon': 144.96444294, 'lat': -37.81057846}"
3,662220240408,66,2024-04-08,22,79,59,138,QVN_T,"{'lon': 144.96444294, 'lat': -37.81057846}"
4,662120240726,66,2024-07-26,21,1034,798,1832,QVN_T,"{'lon': 144.96444294, 'lat': -37.81057846}"


### Extract Geographic Coordinates

The location field contains nested longitude and latitude values.  
This step extracts them into separate columns for easier analysis.

In [6]:
# -----------------------------
# 3A. Extract location coordinates
# -----------------------------
import ast

def extract_coords(x):
    try:
        if pd.isna(x):
            return pd.Series([None, None])
        loc = ast.literal_eval(x) if isinstance(x, str) else x
        return pd.Series([loc.get("lat"), loc.get("lon")])
    except:
        return pd.Series([None, None])

pedestrian_clean[["latitude", "longitude"]] = pedestrian_clean["location"].apply(extract_coords)

print("Missing latitude:", pedestrian_clean["latitude"].isna().sum())
print("Missing longitude:", pedestrian_clean["longitude"].isna().sum())

pedestrian_clean.head()

Missing latitude: 0
Missing longitude: 0


,id,location_id,sensing_date,hourday,direction_1,direction_2,pedestriancount,sensor_name,location,latitude,longitude
0,661720250602,66,2025-06-02,17,1215,1516,2731,QVN_T,"{'lon': 144.96444294, 'lat': -37.81057846}",-37.810578,144.964443
1,661320240712,66,2024-07-12,13,1406,1841,3247,QVN_T,"{'lon': 144.96444294, 'lat': -37.81057846}",-37.810578,144.964443
2,661120250310,66,2025-03-10,11,763,893,1656,QVN_T,"{'lon': 144.96444294, 'lat': -37.81057846}",-37.810578,144.964443
3,662220240408,66,2024-04-08,22,79,59,138,QVN_T,"{'lon': 144.96444294, 'lat': -37.81057846}",-37.810578,144.964443
4,662120240726,66,2024-07-26,21,1034,798,1832,QVN_T,"{'lon': 144.96444294, 'lat': -37.81057846}",-37.810578,144.964443


### Finalise Pedestrian Dataset

Remove redundant columns and retain only useful features for analysis.

In [7]:
# -----------------------------
# 3B. Finalise pedestrian dataset
# -----------------------------
pedestrian_clean = pedestrian_clean.drop(columns=["location"])

print("Final pedestrian columns:")
print(list(pedestrian_clean.columns))

pedestrian_clean.head()

Final pedestrian columns:
['id', 'location_id', 'sensing_date', 'hourday', 'direction_1', 'direction_2', 'pedestriancount', 'sensor_name', 'latitude', 'longitude']


,id,location_id,sensing_date,hourday,direction_1,direction_2,pedestriancount,sensor_name,latitude,longitude
0,661720250602,66,2025-06-02,17,1215,1516,2731,QVN_T,-37.810578,144.964443
1,661320240712,66,2024-07-12,13,1406,1841,3247,QVN_T,-37.810578,144.964443
2,661120250310,66,2025-03-10,11,763,893,1656,QVN_T,-37.810578,144.964443
3,662220240408,66,2024-04-08,22,79,59,138,QVN_T,-37.810578,144.964443
4,662120240726,66,2024-07-26,21,1034,798,1832,QVN_T,-37.810578,144.964443


### Create Timestamp

Combine the date and hour fields to create a complete timestamp for each observation.  
This is required for time-based analysis and aggregation.

In [8]:
# -----------------------------
# 3C. Create timestamp
# -----------------------------
pedestrian_clean["timestamp"] = (
    pedestrian_clean["sensing_date"] +
    pd.to_timedelta(pedestrian_clean["hourday"], unit="h")
)

print("Missing timestamps:", pedestrian_clean["timestamp"].isna().sum())

pedestrian_clean[["sensing_date", "hourday", "timestamp"]].head()

Missing timestamps: 0


,sensing_date,hourday,timestamp
0,2025-06-02,17,2025-06-02 17:00:00
1,2024-07-12,13,2024-07-12 13:00:00
2,2025-03-10,11,2025-03-10 11:00:00
3,2024-04-08,22,2024-04-08 22:00:00
4,2024-07-26,21,2024-07-26 21:00:00


### Observation

A complete timestamp was successfully created by combining the sensing date and hour fields.  
This produced valid time-based records with no missing values, making the pedestrian dataset ready for temporal analysis and hourly aggregation.

## Clean Bicycle Dataset
Remove fully empty columns and retain the most relevant mobility, spatial, and time-based features for later analysis.

In [9]:
# -----------------------------
# 4. Clean bicycle dataset
# -----------------------------
bicycle_clean = bicycle_df.copy()
bicycle_clean.columns = bicycle_clean.columns.str.strip().str.lower()

print("Original shape:", bicycle_clean.shape)

# Identify fully empty columns
empty_columns = bicycle_clean.columns[bicycle_clean.isna().all()]
print("\nCompletely empty columns:", list(empty_columns))

# Drop them
bicycle_clean = bicycle_clean.drop(columns=empty_columns)

print("\nShape after dropping empty columns:", bicycle_clean.shape)

Original shape: (352, 82)

Completely empty columns: ['layout_6', 'layout_6_enter', 'leg1_6', 'leg2_6', 'leg3_6', 'leg4_6', 'leg5_6', 'leg6_1', 'leg6_2', 'leg6_3', 'leg6_4', 'leg6_5', 'leg6_enter', 'leg6_exit', 'leg6_total']

Shape after dropping empty columns: (352, 67)


### Observation

Several columns in the bicycle dataset contained no data across all records.  
These columns were removed as they do not contribute any useful information to the analysis.

This reduced the dataset from 82 to 67 columns, simplifying the structure and improving data quality for further processing.

### Select Relevant Features

The bicycle dataset contains many directional and layout-related columns that are not required for this stage of analysis.  
This step retains only the most relevant features, including location, total counts, and time-based observations.

In [10]:
# -----------------------------
# 4A. Select relevant columns
# -----------------------------
selected_columns = [
    "site_id", "latitude", "longitude", "description",
    "female", "male", "not_known", "total", "year",
    "7_00_am", "7_15_am", "7_30_am", "7_45_am",
    "8_00_am", "8_15_am", "8_30_am", "8_45_am",
    "location", "geolocation"
]

# Keep only existing columns
selected_columns = [col for col in selected_columns if col in bicycle_clean.columns]

bicycle_clean = bicycle_clean[selected_columns]

print("Shape after selecting relevant columns:", bicycle_clean.shape)
bicycle_clean.head()

Shape after selecting relevant columns: (352, 19)


,site_id,latitude,longitude,description,female,male,not_known,total,year,7_00_am,7_15_am,7_30_am,7_45_am,8_00_am,8_15_am,8_30_am,8_45_am,location,geolocation
0,4405,-37.793993,144.941956,"Melrose St [N], Melrose St [S], Mark St [W]",NaN,NaN,20,20,2010,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (144.941956 -37.793993),"{'lon': 144.941956, 'lat': -37.793993}"
1,4417,-37.808903,144.966278,"La Trobe St [E], Russell St (city) [S], La Tro...",NaN,NaN,437,437,2010,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (144.966278 -37.808903),"{'lon': 144.966278, 'lat': -37.808903}"
2,4438,-37.818619,144.970248,"Princes Walk [E], Low Path to river bank [SW],...",NaN,NaN,397,397,2010,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (144.970248 -37.818619),"{'lon': 144.970248, 'lat': -37.818619}"
3,4405,-37.793993,144.941956,"Melrose St [N], Melrose St [S], Mark St [W]",NaN,NaN,16,16,2011,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (144.941956 -37.793993),"{'lon': 144.941956, 'lat': -37.793993}"
4,4443,-37.821100,144.955046,"Flinders St towards Spring St [E], Spencer St ...",NaN,NaN,237,237,2011,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,POINT (144.955046 -37.8211),"{'lon': 144.955046, 'lat': -37.8211}"


### Observation

The cleaned bicycle dataset contains significantly fewer features and a more manageable structure.

However, several columns such as gender breakdown and time-slot counts contain missing values, particularly for earlier years.  
This indicates that detailed time-based data was not consistently collected across all years.

For this reason, time-based analysis will focus on records where time-slot data is available.

### Identify Time-Based Records

This step checks which records contain valid time-slot data, as not all entries include hourly observations.

In [11]:
# -----------------------------
# 4B. Check time column completeness
# -----------------------------
time_columns = [col for col in bicycle_clean.columns if "_am" in col or "_pm" in col]

print("Time columns:", time_columns)

# Check how many rows have at least one time value
bicycle_clean["has_time_data"] = bicycle_clean[time_columns].notna().any(axis=1)

print("\nRows with time data:", bicycle_clean["has_time_data"].sum())
print("Rows without time data:", (~bicycle_clean["has_time_data"]).sum())

Time columns: ['7_00_am', '7_15_am', '7_30_am', '7_45_am', '8_00_am', '8_15_am', '8_30_am', '8_45_am']

Rows with time data: 138
Rows without time data: 214


### Observation

Only a subset of the bicycle dataset contains time-based observations.  
Out of 352 records, 138 contain valid time-slot data, while 214 records do not include hourly information.

To ensure accurate aggregation, only records with valid time data will be used for time-based analysis.

### Filter Time-Based Records

This step filters the dataset to retain only rows that contain valid time-slot data for further processing.

In [12]:
# -----------------------------
# 4C. Filter rows with time data
# -----------------------------
bicycle_time_df = bicycle_clean[bicycle_clean["has_time_data"]].copy()

print("Filtered dataset shape:", bicycle_time_df.shape)
bicycle_time_df.head()

Filtered dataset shape: (138, 20)


,site_id,latitude,longitude,description,female,male,not_known,total,year,7_00_am,7_15_am,7_30_am,7_45_am,8_00_am,8_15_am,8_30_am,8_45_am,location,geolocation,has_time_data
31,4412,-37.799747,144.957474,"Royal Pde [N], Grattan St [E], Royal Pde (city...",475.0,859.0,20,1354,2015,64.0,92.0,105.0,160.0,185.0,192.0,240.0,316.0,POINT (144.957474 -37.799747),"{'lon': 144.957474, 'lat': -37.799747}",True
32,4413,-37.800289,144.943954,"Arden St towards Curzon St [E], Drvyburgh St [...",146.0,295.0,1,442,2015,18.0,30.0,53.0,48.0,96.0,61.0,87.0,49.0,POINT (144.943954 -37.800289),"{'lon': 144.943954, 'lat': -37.800289}",True
33,4414,-37.804562,144.963226,"Swanston St [N], Queensberry St [E], Swanston ...",411.0,659.0,5,1075,2015,32.0,83.0,86.0,124.0,174.0,178.0,226.0,172.0,POINT (144.963226 -37.804562),"{'lon': 144.963226, 'lat': -37.804562}",True
34,4418,-37.809248,144.972971,"Nicholson St [N], Albert St [E], Nicholson St ...",198.0,437.0,1,636,2015,28.0,37.0,61.0,89.0,103.0,107.0,109.0,102.0,POINT (144.972971 -37.809248),"{'lon': 144.972971, 'lat': -37.809248}",True
35,4441,-37.820346,144.957546,"Flinders St towards Spring St [E], King St [S]...",40.0,125.0,5,170,2015,16.0,13.0,21.0,19.0,28.0,24.0,23.0,26.0,POINT (144.957546 -37.820346),"{'lon': 144.957546, 'lat': -37.820346}",True


### Reshape Time Data

Convert time-slot columns into a long format to support aggregation and time-based analysis.

In [13]:
# -----------------------------
# 4D. Reshape time data (wide → long)
# -----------------------------
bicycle_hourly = bicycle_time_df.melt(
    id_vars=[col for col in bicycle_time_df.columns if col not in time_columns],
    value_vars=time_columns,
    var_name="time_slot",
    value_name="count"
)

print("Reshaped dataset shape:", bicycle_hourly.shape)
bicycle_hourly.head()

Reshaped dataset shape: (1104, 14)


,site_id,latitude,longitude,description,female,male,not_known,total,year,location,geolocation,has_time_data,time_slot,count
0,4412,-37.799747,144.957474,"Royal Pde [N], Grattan St [E], Royal Pde (city...",475.0,859.0,20,1354,2015,POINT (144.957474 -37.799747),"{'lon': 144.957474, 'lat': -37.799747}",True,7_00_am,64.0
1,4413,-37.800289,144.943954,"Arden St towards Curzon St [E], Drvyburgh St [...",146.0,295.0,1,442,2015,POINT (144.943954 -37.800289),"{'lon': 144.943954, 'lat': -37.800289}",True,7_00_am,18.0
2,4414,-37.804562,144.963226,"Swanston St [N], Queensberry St [E], Swanston ...",411.0,659.0,5,1075,2015,POINT (144.963226 -37.804562),"{'lon': 144.963226, 'lat': -37.804562}",True,7_00_am,32.0
3,4418,-37.809248,144.972971,"Nicholson St [N], Albert St [E], Nicholson St ...",198.0,437.0,1,636,2015,POINT (144.972971 -37.809248),"{'lon': 144.972971, 'lat': -37.809248}",True,7_00_am,28.0
4,4441,-37.820346,144.957546,"Flinders St towards Spring St [E], King St [S]...",40.0,125.0,5,170,2015,POINT (144.957546 -37.820346),"{'lon': 144.957546, 'lat': -37.820346}",True,7_00_am,16.0


### Observation

The dataset was successfully reshaped from wide to long format.  
Each time-slot column is now represented as a separate row, resulting in 1104 records.

This transformation enables easier time-based analysis and aggregation.

### Clean Time Labels

Standardise the time-slot format for better readability and processing.

In [14]:
# -----------------------------
# 4E. Clean time labels
# -----------------------------
bicycle_hourly["time_slot"] = (
    bicycle_hourly["time_slot"]
    .str.replace("_", " ", regex=False)
    .str.upper()
)

# Remove helper column (no longer needed)
bicycle_hourly = bicycle_hourly.drop(columns=["has_time_data"])

bicycle_hourly.head()

,site_id,latitude,longitude,description,female,male,not_known,total,year,location,geolocation,time_slot,count
0,4412,-37.799747,144.957474,"Royal Pde [N], Grattan St [E], Royal Pde (city...",475.0,859.0,20,1354,2015,POINT (144.957474 -37.799747),"{'lon': 144.957474, 'lat': -37.799747}",7 00 AM,64.0
1,4413,-37.800289,144.943954,"Arden St towards Curzon St [E], Drvyburgh St [...",146.0,295.0,1,442,2015,POINT (144.943954 -37.800289),"{'lon': 144.943954, 'lat': -37.800289}",7 00 AM,18.0
2,4414,-37.804562,144.963226,"Swanston St [N], Queensberry St [E], Swanston ...",411.0,659.0,5,1075,2015,POINT (144.963226 -37.804562),"{'lon': 144.963226, 'lat': -37.804562}",7 00 AM,32.0
3,4418,-37.809248,144.972971,"Nicholson St [N], Albert St [E], Nicholson St ...",198.0,437.0,1,636,2015,POINT (144.972971 -37.809248),"{'lon': 144.972971, 'lat': -37.809248}",7 00 AM,28.0
4,4441,-37.820346,144.957546,"Flinders St towards Spring St [E], King St [S]...",40.0,125.0,5,170,2015,POINT (144.957546 -37.820346),"{'lon': 144.957546, 'lat': -37.820346}",7 00 AM,16.0


### Extract Hour

Extract the hour component from the time-slot field to enable hourly aggregation.

In [15]:
# -----------------------------
# 4F. Extract hour
# -----------------------------
bicycle_hourly["hour"] = (
    bicycle_hourly["time_slot"]
    .str.extract(r"(\d{1,2})")
    .astype(int)
)

print("Unique hours:", bicycle_hourly["hour"].unique())
bicycle_hourly[["time_slot", "hour"]].head()

Unique hours: [7 8]


,time_slot,hour
0,7 00 AM,7
1,7 00 AM,7
2,7 00 AM,7
3,7 00 AM,7
4,7 00 AM,7


### Hourly Aggregation

Aggregate bicycle counts at an hourly level to prepare the dataset for further analysis.

In [16]:
# -----------------------------
# 4G. Aggregate by hour
# -----------------------------
bicycle_aggregated = (
    bicycle_hourly
    .groupby(["year", "hour"], as_index=False)["count"]
    .sum()
)

# Convert count to integer
bicycle_aggregated["count"] = bicycle_aggregated["count"].astype(int)

print("Aggregated shape:", bicycle_aggregated.shape)
bicycle_aggregated.head()

Aggregated shape: (6, 3)


,year,hour,count
0,2015,7,13938
1,2015,8,23701
2,2016,7,13651
3,2016,8,22814
4,2017,7,13920


### Observation

The time-slot data was successfully standardised and transformed into a numerical hour format.  
The dataset contains observations primarily for morning hours (7 AM and 8 AM).

After aggregation, the bicycle dataset provides total counts grouped by year and hour.  
This structured format is suitable for time-based analysis and integration with other datasets.

## Save Cleaned Datasets

Save the cleaned and processed datasets for further analysis and integration in the TARP project.

In [17]:
# -----------------------------
# 5. Save cleaned datasets
# -----------------------------
from pathlib import Path

CLEAN_DATA_DIR = Path("data/processed")
CLEAN_DATA_DIR.mkdir(parents=True, exist_ok=True)

pedestrian_clean_file = CLEAN_DATA_DIR / "pedestrian_clean.csv"
bicycle_clean_file = CLEAN_DATA_DIR / "bicycle_clean.csv"
bicycle_hourly_file = CLEAN_DATA_DIR / "bicycle_hourly.csv"

pedestrian_clean.to_csv(pedestrian_clean_file, index=False)
bicycle_clean.to_csv(bicycle_clean_file, index=False)
bicycle_aggregated.to_csv(bicycle_hourly_file, index=False)

print("Saved pedestrian:", pedestrian_clean_file)
print("Saved bicycle (clean):", bicycle_clean_file)
print("Saved bicycle (hourly):", bicycle_hourly_file)

Saved pedestrian: data/processed/pedestrian_clean.csv
Saved bicycle (clean): data/processed/bicycle_clean.csv
Saved bicycle (hourly): data/processed/bicycle_hourly.csv


In [18]:
# -----------------------------
# 6. Final summary
# -----------------------------
summary = {
    "Pedestrian cleaned rows": len(pedestrian_clean),
    "Bicycle cleaned rows": len(bicycle_clean),
    "Bicycle hourly rows": len(bicycle_aggregated),
    "Pedestrian file": str(pedestrian_clean_file),
    "Bicycle file": str(bicycle_clean_file),
    "Bicycle hourly file": str(bicycle_hourly_file),
}

for key, value in summary.items():
    print(f"{key}: {value}")

files.download(str(pedestrian_clean_file))
files.download(str(bicycle_clean_file))
files.download(str(bicycle_hourly_file))

Pedestrian cleaned rows: 5000
Bicycle cleaned rows: 352
Bicycle hourly rows: 6
Pedestrian file: data/processed/pedestrian_clean.csv
Bicycle file: data/processed/bicycle_clean.csv
Bicycle hourly file: data/processed/bicycle_hourly.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Final Summary

The pedestrian dataset required minimal cleaning and was enhanced with timestamp and location features.

The bicycle dataset required more extensive preprocessing, including removal of empty columns, feature selection, handling incomplete time data, reshaping, and hourly aggregation.

The cleaned datasets are now structured, consistent, and ready for further analysis and integration in the Traffic Accident Risk Prediction (TARP) project.